# Refuerzo: Fundamentos de Clasificación — Dataset Iris
## Ingeniería del Conocimiento (ISO56B) — UNCP
### Msc. Jaime Antonio Huaytalla Pariona — Semestre 2026-II

**Objetivo:** Comprender paso a paso los conceptos fundamentales de clasificación en Machine Learning, desde la exploración de datos hasta la evaluación de modelos.

**Nivel:** Básico → Intermedio

**Contenido:**
1. ¿Qué es clasificación? Exploración visual
2. Train/Test Split: ¿Por qué separar datos?
3. Primer modelo: Regresión Logística
4. Segundo modelo: Árbol de Decisión
5. Tercer modelo: KNN
6. Métricas de evaluación explicadas
7. Comparación de modelos



## 1. Configuración del Entorno

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("✅ Librerías cargadas correctamente")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")


## 2. Cargar y Explorar el Dataset

El dataset **Iris** es el más clásico en Machine Learning. Contiene mediciones de 150 flores de 3 especies:
- **Setosa** (0)
- **Versicolor** (1)
- **Virginica** (2)

Cada flor tiene 4 mediciones (features):
- Largo del sépalo, Ancho del sépalo
- Largo del pétalo, Ancho del pétalo


In [ ]:
# Cargar el dataset
iris = load_iris()

# Convertir a DataFrame para facilitar la exploración
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['especie'] = iris.target
df['nombre_especie'] = df['especie'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Vista general
print("="*60)
print("EXPLORACIÓN DEL DATASET IRIS")
print("="*60)
print(f"\nDimensiones: {df.shape[0]} filas × {df.shape[1]-2} features")
print(f"Clases: {iris.target_names}")
print(f"\nDistribución de clases:")
print(df['nombre_especie'].value_counts())
print(f"\nPrimeras 5 filas:")
df.head()


In [ ]:
# Estadísticas descriptivas por especie
print("Estadísticas por especie:")
print("="*60)
df.groupby('nombre_especie')[iris.feature_names].describe().round(2)


### 2.1 Visualización: ¿Se pueden distinguir las clases?

Antes de aplicar cualquier algoritmo, es FUNDAMENTAL visualizar los datos. Esto nos da intuición sobre qué tan fácil o difícil será clasificarlos.

In [ ]:
# Gráfico 1: Pairplot — Todas las combinaciones de features
fig = sns.pairplot(df, hue='nombre_especie', palette='Set2',
                   vars=iris.feature_names, height=2.5,
                   plot_kws={'alpha': 0.7, 'edgecolor': 'white', 'linewidth': 0.5})
fig.fig.suptitle('Relación entre Features por Especie', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

print("\n📌 OBSERVACIONES:")
print("  - Setosa (verde) se separa fácilmente de las otras dos especies")
print("  - Versicolor y Virginica se superponen parcialmente")
print("  - El largo y ancho del pétalo parecen ser los features más discriminantes")


In [ ]:
# Gráfico 2: Las 2 features más importantes (pétalos)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'setosa': '#66c2a5', 'versicolor': '#fc8d62', 'virginica': '#8da0cb'}

for especie, color in colors.items():
    mask = df['nombre_especie'] == especie
    axes[0].scatter(df.loc[mask, 'petal length (cm)'],
                    df.loc[mask, 'petal width (cm)'],
                    c=color, label=especie, alpha=0.7, edgecolors='white', s=60)

axes[0].set_xlabel('Largo del pétalo (cm)')
axes[0].set_ylabel('Ancho del pétalo (cm)')
axes[0].set_title('Pétalos: Largo vs Ancho')
axes[0].legend()

# Boxplots de cada feature
df_melted = df.melt(id_vars='nombre_especie', value_vars=iris.feature_names,
                     var_name='Feature', value_name='Valor (cm)')
sns.boxplot(data=df_melted, x='Feature', y='Valor (cm)', hue='nombre_especie',
            palette=colors, ax=axes[1])
axes[1].set_title('Distribución de cada Feature por Especie')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print("📌 Los pétalos separan mejor las clases que los sépalos.")
print("   Esto nos dice que un clasificador debería prestar más atención a los pétalos.")


## 3. Preparar los Datos para Machine Learning

### 3.1 ¿Por qué separamos en Train y Test?

Imagina que estudias para un examen usando las preguntas exactas del examen. Sacarías 20/20... pero ¿realmente aprendiste?

Lo mismo pasa con los modelos. Si evaluamos con los mismos datos que usamos para entrenar, no sabemos si el modelo **aprendió** o solo **memorizó**.

In [ ]:
from sklearn.model_selection import train_test_split

# Definir X (features) e y (target)
X = iris.data  # Las 4 mediciones
y = iris.target  # La especie (0, 1, 2)

# Separar: 80% para entrenar, 20% para evaluar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos totales: {len(X)}")
print(f"Datos de entrenamiento: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Datos de prueba: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

print(f"\nDistribución en entrenamiento:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Clase {iris.target_names[u]}: {c} muestras")

print(f"\nDistribución en prueba:")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Clase {iris.target_names[u]}: {c} muestras")

print("\n📌 stratify=y asegura que la proporción de clases se mantiene en ambos conjuntos")


### 3.2 Estandarización: Poner las features en la misma escala

Algunos algoritmos (como SVM y KNN) son sensibles a la **escala** de las features. Si una feature va de 0 a 7 y otra de 0 a 2.5, el algoritmo le da más importancia a la primera.

**StandardScaler** transforma cada feature para que tenga media = 0 y desviación estándar = 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# IMPORTANTE: fit_transform solo en TRAIN, transform en TEST
X_train_scaled = scaler.fit_transform(X_train)  # Aprende media y std del train
X_test_scaled = scaler.transform(X_test)          # Usa la media y std del train

# Verificar la estandarización
print("Antes de estandarizar (train):")
print(f"  Media por feature: {X_train.mean(axis=0).round(2)}")
print(f"  Std por feature:   {X_train.std(axis=0).round(2)}")

print("\nDespués de estandarizar (train):")
print(f"  Media por feature: {X_train_scaled.mean(axis=0).round(4)}")
print(f"  Std por feature:   {X_train_scaled.std(axis=0).round(4)}")

print("\n⚠️ REGLA DE ORO: Nunca hacer fit_transform en el test set.")
print("   Eso causaría 'data leakage' — información del test contamina el entrenamiento.")


## 4. Modelo 1: Regresión Logística

**¿Qué hace?** Calcula una combinación lineal de las features y la pasa por una función sigmoid para obtener probabilidades.

**¿Por qué empezar con esto?** Es el modelo más simple de clasificación. Sirve como **baseline** (punto de referencia) para comparar con modelos más complejos.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Crear y entrenar el modelo
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predecir
y_pred_lr = log_reg.predict(X_test_scaled)

# Evaluar
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy en test: {accuracy_lr:.4f} ({accuracy_lr*100:.1f}%)")
print(f"\nReporte detallado:")
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names))


In [ ]:
# Ver las probabilidades (no solo la clase predicha)
y_proba_lr = log_reg.predict_proba(X_test_scaled)

print("Probabilidades para las primeras 5 muestras de test:")
print(f"{'Setosa':>10} {'Versicolor':>12} {'Virginica':>12}  {'Pred':>10}  {'Real':>10}")
print("-" * 65)
for i in range(5):
    pred = iris.target_names[y_pred_lr[i]]
    real = iris.target_names[y_test[i]]
    match = "✓" if pred == real else "✗"
    print(f"{y_proba_lr[i][0]:>10.4f} {y_proba_lr[i][1]:>12.4f} {y_proba_lr[i][2]:>12.4f}"
          f"  {pred:>10}  {real:>10} {match}")

print("\n📌 El modelo no solo dice la clase, también dice qué tan seguro está.")
print("   Un valor cercano a 1.0 = alta confianza. Cercano a 0.33 = muy inseguro.")


## 5. Modelo 2: Árbol de Decisión

**¿Qué hace?** Clasifica haciendo preguntas secuenciales sobre las features, como un diagrama de flujo.

**Ventaja clave:** Es el modelo más interpretable — puedes visualizar exactamente cómo toma decisiones.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Crear y entrenar el modelo (con restricciones para evitar overfitting)
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)  # OJO: El árbol NO necesita estandarización

# Predecir y evaluar
y_pred_tree = arbol.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)
print(f"Accuracy en test: {accuracy_tree:.4f} ({accuracy_tree*100:.1f}%)")
print(f"\nReporte detallado:")
print(classification_report(y_test, y_pred_tree, target_names=iris.target_names))


In [ ]:
# Visualizar el árbol de decisión
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(arbol,
          feature_names=iris.feature_names,
          class_names=iris.target_names,
          filled=True,
          rounded=True,
          fontsize=11,
          ax=ax)
plt.title('Árbol de Decisión para Clasificar Flores Iris', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

print("📌 CÓMO LEER EL ÁRBOL:")
print("  - Cada nodo muestra la pregunta que hace (ej: 'petal width <= 0.8')")
print("  - 'gini': medida de impureza (0 = puro, 0.5 = muy mezclado)")
print("  - 'samples': cuántas muestras llegan a ese nodo")
print("  - 'value': distribución de clases [setosa, versicolor, virginica]")
print("  - 'class': la clase predicha (la mayoritaria en ese nodo)")


In [ ]:
# Importancia de cada feature según el árbol
importances = pd.Series(arbol.feature_importances_, index=iris.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Importancia')
ax.set_title('Importancia de Features según el Árbol de Decisión')
for i, v in enumerate(importances):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.show()

print("📌 El largo y ancho del pétalo son las features más importantes,")
print("   confirmando lo que vimos en la exploración visual.")


## 6. Modelo 3: K-Nearest Neighbors (KNN)

**¿Qué hace?** Para clasificar un punto nuevo, busca los K puntos más cercanos del entrenamiento y asigna la clase mayoritaria.

**Pregunta clave:** ¿Cuántos vecinos (K) debemos usar?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Probar diferentes valores de K para encontrar el óptimo
k_range = range(1, 21)
train_scores = []
test_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Graficar el efecto de K
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, train_scores, 'o-', label='Train', color='#2196F3')
ax.plot(k_range, test_scores, 's-', label='Test', color='#FF5722')
ax.axvline(x=k_range[np.argmax(test_scores)], color='green', linestyle='--',
           label=f'Mejor K = {k_range[np.argmax(test_scores)]}')
ax.set_xlabel('K (número de vecinos)')
ax.set_ylabel('Accuracy')
ax.set_title('Efecto de K en KNN: ¿Cuántos vecinos usar?')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(k_range)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(test_scores)]
print(f"📌 Mejor K encontrado: {best_k}")
print(f"   K=1: Accuracy train = {train_scores[0]:.3f} (memorizó, overfitting)")
print(f"   K={best_k}: Accuracy test = {max(test_scores):.3f} (generaliza mejor)")


In [ ]:
# Entrenar con el mejor K
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)

y_pred_knn = knn_best.predict(X_test_scaled)
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"KNN (K={best_k}) — Accuracy en test: {accuracy_knn:.4f}")
print(f"\nReporte detallado:")
print(classification_report(y_test, y_pred_knn, target_names=iris.target_names))


## 7. Métricas de Evaluación Explicadas

### 7.1 Matriz de Confusión

La matriz de confusión muestra **exactamente dónde acierta y falla** el modelo.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Matrices de confusión para los 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

modelos_pred = [
    ('Regresión Logística', y_pred_lr),
    ('Árbol de Decisión', y_pred_tree),
    (f'KNN (K={best_k})', y_pred_knn)
]

for ax, (nombre, y_pred) in zip(axes, modelos_pred):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{nombre}\nAccuracy: {acc:.2%}')

plt.suptitle('Matrices de Confusión — Comparación de Modelos', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("📌 CÓMO LEER LA MATRIZ:")
print("  - Diagonal principal (esquina sup-izq a inf-der): ACIERTOS")
print("  - Fuera de la diagonal: ERRORES")
print("  - Ideal: todos los valores en la diagonal y ceros fuera")


## 8. Comparación Final de Modelos

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Definir modelos con pipelines (incluyendo estandarización donde sea necesario)
modelos = {
    'Regresión Logística': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=3, random_state=42),
    f'KNN (K={best_k})': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=best_k))
    ])
}

# Cross-validation 5-fold para cada modelo
print("="*60)
print("COMPARACIÓN CON CROSS-VALIDATION (5-fold)")
print("="*60)

resultados = {}
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
    resultados[nombre] = scores
    print(f"\n{nombre}:")
    print(f"  Scores por fold: {scores.round(4)}")
    print(f"  Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(resultados.values(), labels=resultados.keys(), patch_artist=True)
colores = ['#66c2a5', '#fc8d62', '#8da0cb']
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Accuracy (Cross-Validation)')
ax.set_title('Comparación de Modelos con Cross-Validation')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📌 RESUMEN:")
print("  Cross-validation nos da una estimación más confiable del rendimiento")
print("  porque evalúa el modelo en 5 particiones diferentes del dataset.")


## 9. Conclusiones

En este notebook aprendiste:

1. **Explorar datos** antes de modelar — la visualización revela patrones que los números solos no muestran
2. **Separar en train/test** para evaluar honestamente si el modelo generaliza
3. **Estandarizar features** para que todos los modelos trabajen en la misma escala
4. **Tres algoritmos de clasificación:**
   - Regresión Logística: simple, probabilístico, buen baseline
   - Árbol de Decisión: interpretable, visual, no necesita estandarización
   - KNN: intuitivo, el valor de K es crucial
5. **Evaluar con métricas** más allá de accuracy: precision, recall, F1
6. **Cross-validation** para estimaciones más robustas

---
*Material de refuerzo — Ingeniería del Conocimiento (ISO56B) — UNCP*
